# Phase 0: Bonsai-8B + AppWorld on Colab Pro

**目的**: 推論バックエンド確立 → AppWorld 統合 → 5-10 タスク実測 → スコープ決定

## 1. 依存インストール (全部まとめて)

In [ ]:
# AppWorld は pydantic v1 / fastapi 旧版が必要。先にピン留めする
!pip install -q "pydantic==1.10.26" "fastapi==0.110.3"
!pip install -q appworld huggingface_hub openai
!appworld install
!appworld download data

In [ ]:
import os
import json
import re
import subprocess
import time
import requests
from pathlib import Path
from datetime import datetime
from collections import Counter

## 2. GPU 確認

In [ ]:
!nvidia-smi

## 3. Bonsai-8B GGUF ダウンロード + 推論サーバー起動

上流 llama.cpp は Q1_0_g128 未サポート。PrismML フォークが必要。

In [ ]:
# --- Step 3.1: モデルダウンロード ---
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/content/bonsai-8b-gguf")
MODEL_DIR.mkdir(exist_ok=True)

model_path = hf_hub_download(
    repo_id="prism-ml/Bonsai-8B-gguf",
    filename="Bonsai-8B.gguf",
    local_dir=str(MODEL_DIR),
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# --- Step 3.2: PrismML フォーク llama.cpp CUDA ビルド ---
!git clone https://github.com/PrismML-Eng/llama.cpp /content/llama-cpp-prism 2>/dev/null || echo "Already cloned"
%cd /content/llama-cpp-prism
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native 2>&1 | tail -5
!cmake --build build --config Release -j$(nproc) 2>&1 | tail -10
!ls -la build/bin/llama-server 2>/dev/null || echo "BUILD FAILED"

In [ ]:
# --- Step 3.3: サーバー起動 (ポート 8090) ---
# Colab では 8080 が使用済みのため 8090 を使う

LLAMA_SERVER = "/content/llama-cpp-prism/build/bin/llama-server"
MODEL_FILE = str(list(MODEL_DIR.glob("*.gguf"))[0])

# 既存プロセスがあれば殺す
!pkill -9 -f llama-server 2>/dev/null; sleep 2

server_proc = subprocess.Popen(
    [
        LLAMA_SERVER,
        "-m", MODEL_FILE,
        "--host", "0.0.0.0",
        "--port", "8090",
        "-ngl", "99",       # 全レイヤー GPU オフロード
        "-c", "4096",        # コンテキスト長
    ],
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Waiting for server to start...")
time.sleep(20)

# ヘルスチェック
try:
    r = requests.get("http://localhost:8090/health")
    print(f"Server health: {r.json()}")
except Exception as e:
    print(f"Server not ready: {e}")
    !cat /content/server.log | tail -15

In [ ]:
# --- Step 3.4: 応答確認 ---
from openai import OpenAI

BONSAI_PORT = 8090
client = OpenAI(base_url=f"http://localhost:{BONSAI_PORT}/v1", api_key="dummy")

test_prompts = [
    {"role": "user", "content": "What is 15 + 27?"},
    {"role": "user", "content": "Tags: machine learning, neural networks"},
]

for prompt in test_prompts:
    t0 = time.time()
    response = client.chat.completions.create(
        model="bonsai-8b",
        messages=[prompt],
        temperature=0,
        max_tokens=200,
        seed=100,
    )
    elapsed = time.time() - t0
    print(f"\nPrompt: {prompt['content']}")
    print(f"Response: {response.choices[0].message.content}")
    print(f"Time: {elapsed:.2f}s, Tokens: {response.usage}")

## 4. AppWorld セットアップ

In [ ]:
# --- Step 4.1: タスク構造の確認 ---
from appworld import AppWorld, load_task_ids
from appworld.task import Task

dev_task_ids = load_task_ids("dev")
test_task_ids = load_task_ids("test_normal")
print(f"Dev tasks: {len(dev_task_ids)}")
print(f"Test normal tasks: {len(test_task_ids)}")

# 難易度別カウント (dev)
difficulties = Counter()
for tid in dev_task_ids:
    task = Task.load(task_id=tid)
    difficulties[task.ground_truth.metadata["difficulty"]] += 1
for d in sorted(difficulties):
    print(f"  Difficulty {d}: {difficulties[d]} tasks")

In [ ]:
# --- Step 4.2: 最初のタスクの構造を確認 ---
test_task_id = dev_task_ids[0]
with AppWorld(task_id=test_task_id, experiment_name="phase0_test") as world:
    print(f"Task ID: {test_task_id}")
    print(f"Instruction: {world.task.instruction}")
    print(f"Supervisor: {world.task.supervisor}")
    print(f"Difficulty: {world.task.ground_truth.metadata['difficulty']}")
    print(f"Apps: {list(world.task.app_descriptions.keys())}")

## 5. Bonsai × AppWorld 統合テスト

In [ ]:
# --- エージェント関数定義 ---

def call_bonsai(messages: list[dict], max_tokens: int = 3000) -> str:
    """Bonsai LLM を OpenAI 互換 API 経由で呼び出す"""
    response = client.chat.completions.create(
        model="bonsai-8b",
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
        seed=100,
    )
    return response.choices[0].message.content


def extract_code(text: str) -> str | None:
    """LLM 出力から Python コードブロックを抽出"""
    pattern = r"```python\s*\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if "apis." in text:
        return text.strip()
    return None


SYSTEM_PROMPT = """You are an AI assistant that solves tasks by writing Python code.
You have access to APIs via the `apis` object. Write code to accomplish the task.
Always wrap your code in ```python ... ``` blocks.
Keep your code concise and focused on the task."""


def run_baseline_task(task_id: str, max_steps: int = 20) -> dict:
    """ベースライン: 単一 LLM 呼び出しで 1 タスクを実行"""
    result = {
        "task_id": task_id,
        "success": False,
        "steps": 0,
        "turns": [],
        "wall_time": 0,
        "error": None,
    }
    t0 = time.time()
    try:
        with AppWorld(task_id=task_id, experiment_name="phase0_baseline") as world:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Task: {world.task.instruction}\n\n"
                    f"Supervisor: {world.task.supervisor}\n\n"
                    f"Available apps: {list(world.task.app_descriptions.keys())}"
                )},
            ]
            for step in range(max_steps):
                turn_t0 = time.time()
                llm_output = call_bonsai(messages)
                turn_time = time.time() - turn_t0
                code = extract_code(llm_output)
                turn_log = {
                    "step": step,
                    "prompt_messages": len(messages),
                    "llm_output": llm_output,
                    "code_extracted": code is not None,
                    "turn_time": turn_time,
                }
                if code is None:
                    turn_log["observation"] = "ERROR: No code block found in LLM output"
                    messages.append({"role": "assistant", "content": llm_output})
                    messages.append({"role": "user", "content": turn_log["observation"]})
                    result["turns"].append(turn_log)
                    continue
                try:
                    output = world.execute(code)
                    turn_log["observation"] = output
                except Exception as e:
                    output = f"Execution error: {e}"
                    turn_log["observation"] = output
                messages.append({"role": "assistant", "content": llm_output})
                messages.append({"role": "user", "content": f"Output:\n```\n{output}\n```"})
                result["turns"].append(turn_log)
                result["steps"] = step + 1
                if world.task_completed():
                    result["success"] = True
                    break
    except Exception as e:
        result["error"] = str(e)
    result["wall_time"] = time.time() - t0
    return result

In [ ]:
# --- 統合テスト: 1 タスクで動作確認 ---
print("Running integration test...")
test_result = run_baseline_task(dev_task_ids[0], max_steps=5)
print(f"Task: {test_result['task_id']}")
print(f"Success: {test_result['success']}")
print(f"Steps: {test_result['steps']}")
print(f"Wall time: {test_result['wall_time']:.1f}s")
print(f"Error: {test_result['error']}")

for turn in test_result["turns"]:
    print(f"\n--- Step {turn['step']} ({turn['turn_time']:.1f}s) ---")
    print(f"Code extracted: {turn['code_extracted']}")
    obs = turn["observation"][:300] if turn.get("observation") else "N/A"
    print(f"Observation: {obs}")

## 6. パイロット評価 (10 タスク, 難易度 1)

In [ ]:
# --- 難易度 1 タスクを選定 ---
difficulty_1_tasks = []
for tid in dev_task_ids:
    task = Task.load(task_id=tid)
    if task.ground_truth.metadata["difficulty"] == 1:
        difficulty_1_tasks.append(tid)

pilot_tasks = difficulty_1_tasks[:10]
print(f"Pilot tasks (difficulty 1): {len(pilot_tasks)}")
print(pilot_tasks)

In [ ]:
# --- パイロット実行 (チェックポイント付き) ---
RESULTS_DIR = Path("/content/phase0_results")
RESULTS_DIR.mkdir(exist_ok=True)

pilot_results = []
checkpoint_file = RESULTS_DIR / "phase0_pilot.jsonl"

completed_ids = set()
if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        for line in f:
            r = json.loads(line)
            completed_ids.add(r["task_id"])
            pilot_results.append(r)
    print(f"Resuming: {len(completed_ids)} tasks already completed")

for i, task_id in enumerate(pilot_tasks):
    if task_id in completed_ids:
        print(f"[{i+1}/{len(pilot_tasks)}] {task_id} — skipped (already done)")
        continue
    print(f"\n[{i+1}/{len(pilot_tasks)}] Running {task_id}...")
    result = run_baseline_task(task_id, max_steps=20)
    pilot_results.append(result)
    with open(checkpoint_file, "a") as f:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")
    print(f"  Success: {result['success']}, Steps: {result['steps']}, "
          f"Time: {result['wall_time']:.1f}s, Error: {result['error']}")

In [ ]:
# --- 結果サマリ ---
print("\n" + "=" * 60)
print("PHASE 0 PILOT RESULTS")
print("=" * 60)

successes = sum(1 for r in pilot_results if r["success"])
total = len(pilot_results)
times = [r["wall_time"] for r in pilot_results]
steps = [r["steps"] for r in pilot_results]

print(f"Tasks: {total}")
print(f"Success: {successes}/{total} ({successes/total*100:.1f}%)")
print(f"Wall time: mean={sum(times)/len(times):.1f}s, "
      f"median={sorted(times)[len(times)//2]:.1f}s, "
      f"max={max(times):.1f}s")
print(f"Steps: mean={sum(steps)/len(steps):.1f}, max={max(steps)}")

errors = [r["error"] for r in pilot_results if r["error"]]
if errors:
    print(f"\nErrors ({len(errors)}):")
    for e in errors:
        print(f"  - {e[:100]}")

all_turns = [t for r in pilot_results for t in r["turns"]]
code_extracted = sum(1 for t in all_turns if t["code_extracted"])
print(f"\nCode extraction rate: {code_extracted}/{len(all_turns)} "
      f"({code_extracted/len(all_turns)*100:.1f}%)")

## 7. スコープ決定

| 条件 | スコープ |
|---|---|
| 1タスク以上成功 or 全完走 | フル (test_normal 全タスク) |
| 1タスク30分超 | ミディアム (難易度1のみ) or ミニマム (30タスク) |
| エラー停止多発 | 問題切り分け → 修正 → 再実行 |

In [ ]:
report = {
    "timestamp": datetime.now().isoformat(),
    "inference_backend": "llama.cpp (PrismML fork)",
    "gpu": "NVIDIA A100-SXM4-40GB",
    "port": BONSAI_PORT,
    "pilot_tasks": len(pilot_results),
    "successes": successes,
    "success_rate": successes / total if total > 0 else 0,
    "mean_wall_time_seconds": sum(times) / len(times) if times else 0,
    "max_wall_time_seconds": max(times) if times else 0,
    "code_extraction_rate": code_extracted / len(all_turns) if all_turns else 0,
    "scope_decision": "TODO: フル / ミディアム / ミニマム",
    "notes": "TODO: 観察メモ",
}

report_file = RESULTS_DIR / "phase0_report.json"
with open(report_file, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))